In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE")
DATA_DIR = BASE_DIR / "data"
SRC_DIR = BASE_DIR / "src"
OUT_DIR = BASE_DIR / "outputs"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)
print(list(DATA_DIR.iterdir()))


/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data
[PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/X_val.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/X_train.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/dates_train.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/dates_test.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/X_test.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/dates_val.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/scaler_info.json'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/panel_features.parquet'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/data/X_test_2.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/

In [3]:
import numpy as np

X_train = np.load(DATA_DIR / "X_train_4.npy")
X_val = np.load(DATA_DIR / "X_val_4.npy")
X_test = np.load(DATA_DIR / "X_test_4.npy")

print(X_train.shape, X_val.shape, X_test.shape)

(5522, 391, 19) (1258, 391, 19) (1453, 391, 19)


In [4]:
import sys
sys.path.append(str(SRC_DIR))

from vae_module import VAEModule

In [5]:
print("SRC_DIR:", SRC_DIR)
print("Files in src:", list(SRC_DIR.iterdir()))

SRC_DIR: /content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/src
Files in src: [PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/src/mixture_vae.py'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/src/vae_module.py'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/Mixture_VAE/src/__pycache__')]


In [6]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to Runtime > Change runtime type > GPU.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [9]:
from torch.utils.data import TensorDataset, DataLoader
import torch

batch_size = 256
pin = torch.cuda.is_available()

train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32))
test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32))

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    pin_memory=pin,
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=pin,
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=pin,
)

In [48]:
from types import SimpleNamespace

args = SimpleNamespace(
    name="mixture_vae",

    feature=X_train.shape[-1],
    seq_len=X_train.shape[1],
    n_cluster=2,
    hidden_dim=16,

    tau=0.5,
    hard=False,

    reconstruction_on_s=True,
    reconstruction_on_z="p",

    lamda_m=10.0,
    lamda_i=10.0,
    lamda_t=100.0,
    lamda_b=5.0,

    transition="jump",
    jump_mx=None,

    loss_mode="sum",
    loss_clamp=batch_size * X_train.shape[1] * 10,

    s_clamp=5.0,
    eps=1e-8,
    logvar_min=-10.0,
    logvar_max=10.0,

    s_x_type="lstm",
    s_x_lstm_hidden=64,
    s_x_lstm_layers=1,
    s_x_dropout=0.1,

    z_sx_type="mlp",
    z_sx_hiddens=[128, 128],
    z_sx_dropout=0.1,

    x_sz_type="mlp",
    x_sz_hiddens=[128, 128],
    x_sz_dropout=0.1,

    x_z_type="mlp",
    x_z_hiddens=[128, 128],
    x_z_dropout=0.1,
)

print("Feature dim:", args.feature)
print("Sequence length:", args.seq_len)
print("Batch size:", batch_size)
print("Loss clamp:", args.loss_clamp)

Feature dim: 19
Sequence length: 391
Batch size: 256
Loss clamp: 1000960


In [7]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

In [11]:
import itertools

search_space = {
    "lamda_m": [10.0],
    "lamda_i": [10.0],
    "lamda_t": [10.0, 20.0, 30.0, 50.0, 75.0, 100.0],
    "lamda_b": [1.0, 5.0, 10.0],
    "tau": [0.5, 0.7],
}

candidate_configs = [
    {
        "lamda_m": lamda_m,
        "lamda_i": lamda_i,
        "lamda_t": lamda_t,
        "lamda_b": lamda_b,
        "tau": tau,
    }
    for lamda_m, lamda_i, lamda_t, lamda_b, tau in itertools.product(
        search_space["lamda_m"],
        search_space["lamda_i"],
        search_space["lamda_t"],
        search_space["lamda_b"],
        search_space["tau"],
    )
]

print("Number of configs:", len(candidate_configs))
candidate_configs[:5]

Number of configs: 36


[{'lamda_m': 10.0,
  'lamda_i': 10.0,
  'lamda_t': 10.0,
  'lamda_b': 1.0,
  'tau': 0.5},
 {'lamda_m': 10.0,
  'lamda_i': 10.0,
  'lamda_t': 10.0,
  'lamda_b': 1.0,
  'tau': 0.7},
 {'lamda_m': 10.0,
  'lamda_i': 10.0,
  'lamda_t': 10.0,
  'lamda_b': 5.0,
  'tau': 0.5},
 {'lamda_m': 10.0,
  'lamda_i': 10.0,
  'lamda_t': 10.0,
  'lamda_b': 5.0,
  'tau': 0.7},
 {'lamda_m': 10.0,
  'lamda_i': 10.0,
  'lamda_t': 10.0,
  'lamda_b': 10.0,
  'tau': 0.5}]

In [12]:
import copy
import numpy as np
import pandas as pd
import time
import torch

results = []
candidate_models = {}
candidate_histories = {}

for i, cfg in enumerate(candidate_configs, start=1):
    args_run = copy.deepcopy(args)

    args_run.lamda_m = cfg["lamda_m"]
    args_run.lamda_i = cfg["lamda_i"]
    args_run.lamda_t = cfg["lamda_t"]
    args_run.lamda_b = cfg["lamda_b"]
    args_run.tau = cfg["tau"]

    key = (
        cfg["lamda_m"],
        cfg["lamda_i"],
        cfg["lamda_t"],
        cfg["tau"],
        cfg["lamda_b"],
    )

    print("=" * 100)
    print(f"Config {i}/{len(candidate_configs)}")
    print(
        f"Running Mixture-VAE | "
        f"lamda_m={cfg['lamda_m']}, "
        f"lamda_i={cfg['lamda_i']}, "
        f"lamda_t={cfg['lamda_t']}, "
        f"tau={cfg['tau']}, "
        f"lamda_b={cfg['lamda_b']}"
    )

    start_time = time.time()

    vae_run = VAEModule(args_run)

    history_run = vae_run.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        lr=1e-3,
        epochs=40,
        weight_decay=1e-5,
        patience=15,
        verbose=True,
    )

    elapsed_sec = time.time() - start_time

    if len(history_run["val_loss"]) == 0:
        print("No validation loss recorded. Skipping.")
        continue

    best_epoch_idx = int(np.argmin(history_run["val_loss"]))

    best_usage = np.asarray(history_run["val_regime_usage"][best_epoch_idx])
    best_eff = float(history_run["val_effective_regimes"][best_epoch_idx])
    best_switch = float(history_run["val_switch_rate"][best_epoch_idx])

    result = {
        "lamda_m": cfg["lamda_m"],
        "lamda_i": cfg["lamda_i"],
        "lamda_t": cfg["lamda_t"],
        "tau": cfg["tau"],
        "lamda_b": cfg["lamda_b"],

        "best_epoch": best_epoch_idx + 1,
        "elapsed_sec": elapsed_sec,

        "val_total_loss": float(history_run["val_loss"][best_epoch_idx]),
        "val_recon_loss": float(history_run["val_loss_r"][best_epoch_idx]),
        "val_mixture_loss": float(history_run["val_loss_m"][best_epoch_idx]),
        "val_info_loss": float(history_run["val_loss_i"][best_epoch_idx]),
        "val_transition_loss": float(history_run["val_loss_t"][best_epoch_idx]),
        "val_balance_loss": float(history_run["val_loss_b"][best_epoch_idx]),

        "usage_0": float(best_usage[0]),
        "usage_1": float(best_usage[1]),
        "min_usage": float(np.min(best_usage)),
        "max_usage": float(np.max(best_usage)),
        "eff_regimes": best_eff,
        "switch_rate": best_switch,
    }

    result["weighted_mixture"] = result["lamda_m"] * result["val_mixture_loss"]
    result["weighted_info"] = result["lamda_i"] * result["val_info_loss"]
    result["weighted_transition"] = result["lamda_t"] * result["val_transition_loss"]
    result["weighted_balance"] = result["lamda_b"] * result["val_balance_loss"]

    results.append(result)

    candidate_models[key] = vae_run
    candidate_histories[key] = history_run

    results_df = pd.DataFrame(results)
    #results_df.to_csv("mixture_vae_trend_regime_hyperparameter_results.csv", index=False)

    display(
        results_df.sort_values(
            ["min_usage", "switch_rate"],
            ascending=[False, True]
        ).head(10)
    )

Config 1/36
Running Mixture-VAE | lamda_m=10.0, lamda_i=10.0, lamda_t=10.0, tau=0.5, lamda_b=1.0
Epoch [1/40] | train_loss=25054.612414 | val_total=15828.903418 | val_recon=4141.752484 | usage=[0.2575 0.7425] | eff=1.619 | switch=0.0003
Epoch [10/40] | train_loss=14445.108475 | val_total=12282.734897 | val_recon=3188.724414 | usage=[0.404 0.596] | eff=1.929 | switch=0.0522
Epoch [20/40] | train_loss=14178.421700 | val_total=12081.075517 | val_recon=3026.801023 | usage=[0.428 0.572] | eff=1.959 | switch=0.0532
Epoch [30/40] | train_loss=14108.756157 | val_total=12228.199722 | val_recon=3187.100705 | usage=[0.4484 0.5516] | eff=1.979 | switch=0.0528
Epoch [40/40] | train_loss=14056.542195 | val_total=12081.714825 | val_recon=3045.146810 | usage=[0.4382 0.5618] | eff=1.970 | switch=0.0502


,lamda_m,lamda_i,lamda_t,tau,lamda_b,best_epoch,elapsed_sec,val_total_loss,val_recon_loss,val_mixture_loss,...,usage_0,usage_1,min_usage,max_usage,eff_regimes,switch_rate,weighted_mixture,weighted_info,weighted_transition,weighted_balance
0,10.0,10.0,10.0,0.5,1.0,29,21.957585,12000.746025,2959.149444,960.370417,...,0.437714,0.562286,0.437714,0.562286,1.969438,0.05223,9603.704168,-1777.581292,1212.224901,3.248722


Config 2/36
Running Mixture-VAE | lamda_m=10.0, lamda_i=10.0, lamda_t=10.0, tau=0.7, lamda_b=1.0
Epoch [1/40] | train_loss=23740.577553 | val_total=16133.268680 | val_recon=4117.600755 | usage=[0.5689 0.4311] | eff=1.963 | switch=0.0230
Epoch [10/40] | train_loss=14130.114089 | val_total=12133.138315 | val_recon=3089.997913 | usage=[0.4924 0.5076] | eff=2.000 | switch=0.0615
Epoch [20/40] | train_loss=13912.234245 | val_total=11925.015302 | val_recon=2895.488523 | usage=[0.4788 0.5212] | eff=1.996 | switch=0.0675
Epoch [30/40] | train_loss=13809.267928 | val_total=11855.988076 | val_recon=2843.971333 | usage=[0.4832 0.5168] | eff=1.998 | switch=0.0634
Epoch [40/40] | train_loss=13801.741602 | val_total=12056.400437 | val_recon=3049.442915 | usage=[0.4877 0.5123] | eff=1.999 | switch=0.0560


,lamda_m,lamda_i,lamda_t,tau,lamda_b,best_epoch,elapsed_sec,val_total_loss,val_recon_loss,val_mixture_loss,...,usage_0,usage_1,min_usage,max_usage,eff_regimes,switch_rate,weighted_mixture,weighted_info,weighted_transition,weighted_balance
1,10.0,10.0,10.0,0.7,1.0,30,15.705382,11855.988076,2843.971333,960.289721,...,0.483201,0.516799,0.483201,0.516799,1.997745,0.063355,9602.897208,-1868.207286,1277.040922,0.285781
0,10.0,10.0,10.0,0.5,1.0,29,21.957585,12000.746025,2959.149444,960.370417,...,0.437714,0.562286,0.437714,0.562286,1.969438,0.052230,9603.704168,-1777.581292,1212.224901,3.248722


Config 3/36
Running Mixture-VAE | lamda_m=10.0, lamda_i=10.0, lamda_t=10.0, tau=0.5, lamda_b=5.0
Epoch [1/40] | train_loss=25338.717041 | val_total=16761.998410 | val_recon=4095.237729 | usage=[0.752 0.248] | eff=1.595 | switch=0.0009
Epoch [10/40] | train_loss=14377.119069 | val_total=12259.431638 | val_recon=3183.514805 | usage=[0.5406 0.4594] | eff=1.987 | switch=0.0581
Epoch [20/40] | train_loss=14170.210318 | val_total=12168.267687 | val_recon=3113.650040 | usage=[0.5345 0.4655] | eff=1.991 | switch=0.0588
Epoch [30/40] | train_loss=14060.346432 | val_total=12134.588434 | val_recon=3092.906846 | usage=[0.5294 0.4706] | eff=1.993 | switch=0.0577
Epoch [40/40] | train_loss=14004.267543 | val_total=12091.444157 | val_recon=3059.159678 | usage=[0.5304 0.4696] | eff=1.993 | switch=0.0556


...

,lamda_m,lamda_i,lamda_t,tau,lamda_b,best_epoch,elapsed_sec,val_total_loss,val_recon_loss,val_mixture_loss,...,usage_0,usage_1,min_usage,max_usage,eff_regimes,switch_rate,weighted_mixture,weighted_info,weighted_transition,weighted_balance
23,10.0,10.0,50.0,0.7,10.0,39,15.301800,13201.699126,3272.630068,960.109859,...,0.503282,0.496718,0.496718,0.503282,1.999914,0.015711,9601.098594,-115.083908,441.720120,1.334373
4,10.0,10.0,10.0,0.5,10.0,37,15.504573,11979.501987,2954.895866,960.234648,...,0.511318,0.488682,0.488682,0.511318,1.998976,0.063931,9602.346483,-1849.020488,1269.642398,1.637267
16,10.0,10.0,30.0,0.5,10.0,40,15.416002,12966.439587,3055.410374,960.290454,...,0.486738,0.513262,0.486738,0.513262,1.998594,0.020391,9602.904536,-391.274554,696.427637,2.971411
5,10.0,10.0,10.0,0.7,10.0,38,15.407031,11875.041534,2854.414994,960.143469,...,0.514443,0.485557,0.485557,0.514443,1.998333,0.061445,9601.434693,-1846.270432,1263.148583,2.313330
1,10.0,10.0,10.0,0.7,1.0,30,15.705382,11855.988076,2843.971333,960.289721,...,0.483201,0.516799,0.483201,0.516799,1.997745,0.063355,9602.897208,-1868.207286,1277.040922,0.285781
14,10.0,10.0,30.0,0.5,5.0,32,15.269075,12978.403219,3112.882552,960.446964,...,0.521986,0.478014,0.478014,0.521986,1.996140,0.020684,9604.469644,-319.082982,578.018331,2.115743
22,10.0,10.0,50.0,0.5,10.0,23,14.424851,13261.227941,3404.360791,960.351413,...,0.522383,0.477617,0.477617,0.522383,1.996000,0.013330,9603.514135,-66.758273,315.849244,4.262317
8,10.0,10.0,20.0,0.5,5.0,39,15.251406,12688.069555,2861.028418,960.218601,...,0.475777,0.524223,0.475777,0.524223,1.995317,0.027628,9602.186010,-972.697176,1194.396503,3.155333
3,10.0,10.0,10.0,0.7,5.0,34,15.418516,11887.552464,2868.406151,960.397990,...,0.475369,0.524631,0.475369,0.524631,1.995159,0.055020,9603.979904,-1856.012582,1268.322471,2.856841
2,10.0,10.0,10.0,0.5,5.0,28,15.694473,11988.917727,2947.747118,960.441624,...,0.530912,0.469088,0.469088,0.530912,1.992384,0.058137,9604.416236,-1799.129167,1231.572793,4.310936


Config 35/36
Running Mixture-VAE | lamda_m=10.0, lamda_i=10.0, lamda_t=100.0, tau=0.5, lamda_b=10.0
Epoch [1/40] | train_loss=40450.602590 | val_total=28317.292528 | val_recon=3954.271612 | usage=[0.6807 0.3193] | eff=1.769 | switch=0.0292
Epoch [10/40] | train_loss=16060.117711 | val_total=13613.220588 | val_recon=3534.876689 | usage=[0.5854 0.4146] | eff=1.943 | switch=0.0072
Epoch [20/40] | train_loss=15833.153477 | val_total=13549.219595 | val_recon=3470.185215 | usage=[0.6037 0.3963] | eff=1.918 | switch=0.0090
Epoch [30/40] | train_loss=15858.573026 | val_total=13604.170509 | val_recon=3409.324275 | usage=[0.5684 0.4316] | eff=1.963 | switch=0.0131
Early stopping at epoch 34. Best val loss = 13534.087639


,lamda_m,lamda_i,lamda_t,tau,lamda_b,best_epoch,elapsed_sec,val_total_loss,val_recon_loss,val_mixture_loss,...,usage_0,usage_1,min_usage,max_usage,eff_regimes,switch_rate,weighted_mixture,weighted_info,weighted_transition,weighted_balance
23,10.0,10.0,50.0,0.7,10.0,39,15.301800,13201.699126,3272.630068,960.109859,...,0.503282,0.496718,0.496718,0.503282,1.999914,0.015711,9601.098594,-115.083908,441.720120,1.334373
4,10.0,10.0,10.0,0.5,10.0,37,15.504573,11979.501987,2954.895866,960.234648,...,0.511318,0.488682,0.488682,0.511318,1.998976,0.063931,9602.346483,-1849.020488,1269.642398,1.637267
16,10.0,10.0,30.0,0.5,10.0,40,15.416002,12966.439587,3055.410374,960.290454,...,0.486738,0.513262,0.486738,0.513262,1.998594,0.020391,9602.904536,-391.274554,696.427637,2.971411
5,10.0,10.0,10.0,0.7,10.0,38,15.407031,11875.041534,2854.414994,960.143469,...,0.514443,0.485557,0.485557,0.514443,1.998333,0.061445,9601.434693,-1846.270432,1263.148583,2.313330
1,10.0,10.0,10.0,0.7,1.0,30,15.705382,11855.988076,2843.971333,960.289721,...,0.483201,0.516799,0.483201,0.516799,1.997745,0.063355,9602.897208,-1868.207286,1277.040922,0.285781
14,10.0,10.0,30.0,0.5,5.0,32,15.269075,12978.403219,3112.882552,960.446964,...,0.521986,0.478014,0.478014,0.521986,1.996140,0.020684,9604.469644,-319.082982,578.018331,2.115743
22,10.0,10.0,50.0,0.5,10.0,23,14.424851,13261.227941,3404.360791,960.351413,...,0.522383,0.477617,0.477617,0.522383,1.996000,0.013330,9603.514135,-66.758273,315.849244,4.262317
8,10.0,10.0,20.0,0.5,5.0,39,15.251406,12688.069555,2861.028418,960.218601,...,0.475777,0.524223,0.475777,0.524223,1.995317,0.027628,9602.186010,-972.697176,1194.396503,3.155333
3,10.0,10.0,10.0,0.7,5.0,34,15.418516,11887.552464,2868.406151,960.397990,...,0.475369,0.524631,0.475369,0.524631,1.995159,0.055020,9603.979904,-1856.012582,1268.322471,2.856841
2,10.0,10.0,10.0,0.5,5.0,28,15.694473,11988.917727,2947.747118,960.441624,...,0.530912,0.469088,0.469088,0.530912,1.992384,0.058137,9604.416236,-1799.129167,1231.572793,4.310936


Config 36/36
Running Mixture-VAE | lamda_m=10.0, lamda_i=10.0, lamda_t=100.0, tau=0.7, lamda_b=10.0
Epoch [1/40] | train_loss=40650.084028 | val_total=29181.780207 | val_recon=3780.867299 | usage=[0.5808 0.4192] | eff=1.949 | switch=0.0271
Epoch [10/40] | train_loss=16224.081175 | val_total=13761.129769 | val_recon=3530.950666 | usage=[0.5958 0.4042] | eff=1.929 | switch=0.0108
Epoch [20/40] | train_loss=15887.856030 | val_total=13578.553458 | val_recon=3579.267886 | usage=[0.5689 0.4311] | eff=1.963 | switch=0.0080
Epoch [30/40] | train_loss=15854.601820 | val_total=13601.498808 | val_recon=3803.268333 | usage=[0.5093 0.4907] | eff=1.999 | switch=0.0045
Epoch [40/40] | train_loss=15803.981891 | val_total=13559.074722 | val_recon=3654.403219 | usage=[0.5586 0.4414] | eff=1.973 | switch=0.0067


,lamda_m,lamda_i,lamda_t,tau,lamda_b,best_epoch,elapsed_sec,val_total_loss,val_recon_loss,val_mixture_loss,...,usage_0,usage_1,min_usage,max_usage,eff_regimes,switch_rate,weighted_mixture,weighted_info,weighted_transition,weighted_balance
23,10.0,10.0,50.0,0.7,10.0,39,15.301800,13201.699126,3272.630068,960.109859,...,0.503282,0.496718,0.496718,0.503282,1.999914,0.015711,9601.098594,-115.083908,441.720120,1.334373
4,10.0,10.0,10.0,0.5,10.0,37,15.504573,11979.501987,2954.895866,960.234648,...,0.511318,0.488682,0.488682,0.511318,1.998976,0.063931,9602.346483,-1849.020488,1269.642398,1.637267
35,10.0,10.0,100.0,0.7,10.0,35,15.256152,13528.154610,3666.817717,960.138352,...,0.488090,0.511910,0.488090,0.511910,1.998866,0.006261,9601.383520,-28.355838,286.920840,1.388283
16,10.0,10.0,30.0,0.5,10.0,40,15.416002,12966.439587,3055.410374,960.290454,...,0.486738,0.513262,0.486738,0.513262,1.998594,0.020391,9602.904536,-391.274554,696.427637,2.971411
5,10.0,10.0,10.0,0.7,10.0,38,15.407031,11875.041534,2854.414994,960.143469,...,0.514443,0.485557,0.485557,0.514443,1.998333,0.061445,9601.434693,-1846.270432,1263.148583,2.313330
1,10.0,10.0,10.0,0.7,1.0,30,15.705382,11855.988076,2843.971333,960.289721,...,0.483201,0.516799,0.483201,0.516799,1.997745,0.063355,9602.897208,-1868.207286,1277.040922,0.285781
14,10.0,10.0,30.0,0.5,5.0,32,15.269075,12978.403219,3112.882552,960.446964,...,0.521986,0.478014,0.478014,0.521986,1.996140,0.020684,9604.469644,-319.082982,578.018331,2.115743
22,10.0,10.0,50.0,0.5,10.0,23,14.424851,13261.227941,3404.360791,960.351413,...,0.522383,0.477617,0.477617,0.522383,1.996000,0.013330,9603.514135,-66.758273,315.849244,4.262317
8,10.0,10.0,20.0,0.5,5.0,39,15.251406,12688.069555,2861.028418,960.218601,...,0.475777,0.524223,0.475777,0.524223,1.995317,0.027628,9602.186010,-972.697176,1194.396503,3.155333
3,10.0,10.0,10.0,0.7,5.0,34,15.418516,11887.552464,2868.406151,960.397990,...,0.475369,0.524631,0.475369,0.524631,1.995159,0.055020,9603.979904,-1856.012582,1268.322471,2.856841


In [14]:
dates_val = np.load(DATA_DIR / "dates_val_4.npy", allow_pickle=True)
dates_test = np.load(DATA_DIR / "dates_test_4.npy", allow_pickle=True)
panel_features = pd.read_parquet(DATA_DIR / "panel_features_4.parquet")

In [15]:
results_df = pd.DataFrame(results)



def daily_segment_summary(states_2d, dates):
    rows = []

    for i in range(len(dates)):
        s = states_2d[i]

        switches = int(np.sum(s[1:] != s[:-1]))
        segments = switches + 1

        rows.append({
            "date": pd.to_datetime(dates[i]).date(),
            "switches": switches,
            "segments": segments,
        })

    df = pd.DataFrame(rows)

    summary = {
        "median_segments": df["segments"].median(),
        "mean_segments": df["segments"].mean(),
        "p75_segments": df["segments"].quantile(0.75),
        "p90_segments": df["segments"].quantile(0.90),
        "max_segments": df["segments"].max(),
        "one_state_share": (df["segments"] == 1).mean(),
        "multi_state_share": (df["segments"] > 1).mean(),
    }

    return summary, df


daily_rows = []

for _, row in results_df.iterrows():
    key = (
        float(row["lamda_m"]),
        float(row["lamda_i"]),
        float(row["lamda_t"]),
        float(row["tau"]),
        float(row["lamda_b"]),
    )

    model = candidate_models[key]

    # Use validation set for model selection.
    states_val = model.get_predicted_states(val_loader)

    daily_summary, _ = daily_segment_summary(states_val, dates_val)

    daily_rows.append({
        "lamda_m": key[0],
        "lamda_i": key[1],
        "lamda_t": key[2],
        "tau": key[3],
        "lamda_b": key[4],
        **daily_summary,
    })

daily_df = pd.DataFrame(daily_rows)

selection_with_daily = results_df.merge(
    daily_df,
    on=["lamda_m", "lamda_i", "lamda_t", "tau", "lamda_b"],
    how="left"
)

print("Original Mixture-VAE models:", len(results_df))

admissible_df = selection_with_daily[
    (selection_with_daily["min_usage"] >= 0.25) &
    (selection_with_daily["eff_regimes"] >= 1.70) &
    (selection_with_daily["switch_rate"] >= 0.003) &
    (selection_with_daily["switch_rate"] <= 0.050) &
    (selection_with_daily["median_segments"] >= 1) &
    (selection_with_daily["median_segments"] <= 16) &
    (selection_with_daily["p90_segments"] <= 28)
].copy()

print("Admissible Mixture-VAE models under common 2-state criterion:", len(admissible_df))

#Relaxed critiria


if len(admissible_df) == 0:
    print("No strict admissible Mixture-VAE models. Using relaxed 2-state criteria.")

    admissible_df = selection_with_daily[
        (selection_with_daily["min_usage"] >= 0.15) &
        (selection_with_daily["eff_regimes"] >= 1.50) &
        (selection_with_daily["switch_rate"] >= 0.001) &
        (selection_with_daily["switch_rate"] <= 0.070) &
        (selection_with_daily["median_segments"] <= 22) &
        (selection_with_daily["p90_segments"] <= 34)
    ].copy()

    print("Relaxed admissible Mixture-VAE models:", len(admissible_df))

selection_df = admissible_df.sort_values(
    ["val_recon_loss", "val_total_loss"],
    ascending=[True, True]
).copy()

display_cols = [
    "lamda_m", "lamda_i", "lamda_t", "tau", "lamda_b",
    "best_epoch",
    "val_recon_loss", "val_total_loss",
    "min_usage", "eff_regimes", "switch_rate",
    "median_segments", "mean_segments", "p75_segments",
    "p90_segments", "max_segments", "one_state_share",
]

display(selection_df[display_cols].head(15))

Original Mixture-VAE models: 36
Admissible Mixture-VAE models under common 2-state criterion: 24


,lamda_m,lamda_i,lamda_t,tau,lamda_b,best_epoch,val_recon_loss,val_total_loss,min_usage,eff_regimes,switch_rate,median_segments,mean_segments,p75_segments,p90_segments,max_segments,one_state_share
9,10.0,10.0,20.0,0.7,5.0,38,2752.145394,12578.501789,0.417537,1.947040,0.024237,10.0,10.452305,13.0,15.0,25,0.002385
11,10.0,10.0,20.0,0.7,10.0,38,2758.670906,12581.158983,0.445064,1.976144,0.024502,10.0,10.555644,13.0,15.0,23,0.000795
7,10.0,10.0,20.0,0.7,1.0,35,2809.165143,12612.795509,0.418863,1.948686,0.024296,10.0,10.475358,13.0,15.0,22,0.000795
17,10.0,10.0,30.0,0.7,10.0,35,2850.862530,12848.308227,0.426663,1.957879,0.017087,7.0,7.663752,9.0,11.0,21,0.007949
10,10.0,10.0,20.0,0.5,10.0,33,2854.356816,12705.368243,0.450298,1.980431,0.028474,12.0,12.104928,15.0,17.0,29,0.000795
8,10.0,10.0,20.0,0.5,5.0,39,2861.028418,12688.069555,0.475777,1.995317,0.027628,12.0,11.775040,14.0,17.0,26,0.000000
13,10.0,10.0,30.0,0.7,1.0,35,2894.318313,12824.691375,0.337416,1.808753,0.016402,7.0,7.396661,9.0,11.0,18,0.015898
6,10.0,10.0,20.0,0.5,1.0,31,2901.664199,12706.095390,0.426174,1.957328,0.027005,11.0,11.531797,14.0,16.0,30,0.000795
15,10.0,10.0,30.0,0.7,5.0,36,2916.268879,12858.564587,0.451388,1.981272,0.017137,7.0,7.683625,9.0,11.0,19,0.002385
16,10.0,10.0,30.0,0.5,10.0,40,3055.410374,12966.439587,0.486738,1.998594,0.020391,9.0,8.952305,11.0,13.0,20,0.000000


In [16]:
best_row = selection_df.iloc[0]

final_key = (
    float(best_row["lamda_m"]),
    float(best_row["lamda_i"]),
    float(best_row["lamda_t"]),
    float(best_row["tau"]),
    float(best_row["lamda_b"]),
)

print("Selected final key:", final_key)

final_model = candidate_models[final_key]
final_history = candidate_histories[final_key]

Selected final key: (10.0, 10.0, 20.0, 0.7, 5.0)


In [17]:
final_test_states_2d = final_model.get_predicted_states(test_loader)
final_test_states_flat = final_test_states_2d.reshape(-1)

print("Final test states 2D:", final_test_states_2d.shape)
print("Final test states flat:", final_test_states_flat.shape)

Final test states 2D: (1453, 391)
Final test states flat: (568123,)


we will retrain for the final keys for 2 states cause we got disconnected

In [10]:
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
import random

def make_mvae_args(n_cluster, lamda_b):
    return SimpleNamespace(
        name="mixture_vae",
        feature=X_train.shape[-1],
        seq_len=X_train.shape[1],
        n_cluster=n_cluster,

        hidden_dim=16,
        tau=0.7,
        hard=False,

        reconstruction_on_s=True,
        reconstruction_on_z="p",

        lamda_m=10.0,
        lamda_i=10.0,
        lamda_t=20.0,
        lamda_b=lamda_b,

        transition="jump",
        jump_mx=None,

        loss_mode="sum",
        loss_clamp=batch_size * X_train.shape[1] * 10,
        s_clamp=5.0,

        eps=1e-8,
        logvar_min=-10.0,
        logvar_max=10.0,

        s_x_type="lstm",
        s_x_lstm_hidden=64,
        s_x_lstm_layers=1,
        s_x_dropout=0.1,

        z_sx_type="mlp",
        z_sx_hiddens=[128, 128],
        z_sx_dropout=0.1,

        x_sz_type="mlp",
        x_sz_hiddens=[128, 128],
        x_sz_dropout=0.1,

        x_z_type="mlp",
        x_z_hiddens=[128, 128],
        x_z_dropout=0.1,
    )

args_mvae_2 = make_mvae_args(n_cluster=2, lamda_b=5.0)
selected_key_mvae_2 = (10.0, 10.0, 20.0, 0.7, 5.0)
print(args_mvae_2)

namespace(name='mixture_vae', feature=19, seq_len=391, n_cluster=2, hidden_dim=16, tau=0.7, hard=False, reconstruction_on_s=True, reconstruction_on_z='p', lamda_m=10.0, lamda_i=10.0, lamda_t=20.0, lamda_b=5.0, transition='jump', jump_mx=None, loss_mode='sum', loss_clamp=1000960, s_clamp=5.0, eps=1e-08, logvar_min=-10.0, logvar_max=10.0, s_x_type='lstm', s_x_lstm_hidden=64, s_x_lstm_layers=1, s_x_dropout=0.1, z_sx_type='mlp', z_sx_hiddens=[128, 128], z_sx_dropout=0.1, x_sz_type='mlp', x_sz_hiddens=[128, 128], x_sz_dropout=0.1, x_z_type='mlp', x_z_hiddens=[128, 128], x_z_dropout=0.1)


In [11]:
vae_run_2 = VAEModule(args_mvae_2)

history_mvae_2 = vae_run_2.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    lr=1e-3,
    epochs=40,
    weight_decay=1e-5,
    patience=15,
    verbose=True,
)

print("Finished final 2-state Mixture-VAE.")

Epoch [1/40] | train_loss=25628.368707 | val_total=17199.168919 | val_recon=4001.026381 | usage=[0.5539 0.4461] | eff=1.977 | switch=0.0238
Epoch [10/40] | train_loss=14980.661943 | val_total=12833.482909 | val_recon=2975.344098 | usage=[0.5957 0.4043] | eff=1.929 | switch=0.0305
Epoch [20/40] | train_loss=14706.509779 | val_total=12757.815183 | val_recon=2909.870280 | usage=[0.5795 0.4205] | eff=1.951 | switch=0.0256
Epoch [30/40] | train_loss=14617.347315 | val_total=12728.035771 | val_recon=2897.008496 | usage=[0.5682 0.4318] | eff=1.963 | switch=0.0249
Epoch [40/40] | train_loss=14569.658548 | val_total=12695.886924 | val_recon=2876.160622 | usage=[0.5807 0.4193] | eff=1.949 | switch=0.0239
Finished final 2-state Mixture-VAE.


In [12]:
states_train_mvae_2 = vae_run_2.get_predicted_states(train_loader)
states_val_mvae_2   = vae_run_2.get_predicted_states(val_loader)
states_test_mvae_2  = vae_run_2.get_predicted_states(test_loader)

states_train_mvae_2 = np.asarray(states_train_mvae_2).reshape(X_train.shape[0], X_train.shape[1])
states_val_mvae_2   = np.asarray(states_val_mvae_2).reshape(X_val.shape[0], X_val.shape[1])
states_test_mvae_2  = np.asarray(states_test_mvae_2).reshape(X_test.shape[0], X_test.shape[1])

print("train:", states_train_mvae_2.shape)
print("val:", states_val_mvae_2.shape)
print("test:", states_test_mvae_2.shape)

np.save(OUT_DIR / "mvae_2state_train_states.npy", states_train_mvae_2)
np.save(OUT_DIR / "mvae_2state_val_states.npy", states_val_mvae_2)
np.save(OUT_DIR / "mvae_2state_test_states.npy", states_test_mvae_2)

torch.save(
    {
        "selected_key": selected_key_mvae_2,
        "args": vars(args_mvae_2),
        "history": history_mvae_2,
    },
    OUT_DIR / "mvae_2state_selected_config.pt"
)

print("Saved 2-state Mixture-VAE states/config.")

train: (5522, 391)
val: (1258, 391)
test: (1453, 391)
Saved 2-state Mixture-VAE states/config.


In [13]:
states_val_mvae_2 = np.load(OUT_DIR / "mvae_2state_val_states.npy")
states_test_mvae_2 = np.load(OUT_DIR / "mvae_2state_test_states.npy")

print("2-state val:", states_val_mvae_2.shape)
print("2-state test:", states_test_mvae_2.shape)

2-state val: (1258, 391)
2-state test: (1453, 391)


In [14]:
import numpy as np
import pandas as pd

econ_cols = [
    "ret_mean_30", "ret_mean_60", "ret_mean_120",
    "momentum_30", "momentum_60", "momentum_120",
    "trend_strength_30", "trend_strength_60", "trend_strength_120",
    "ret_std_60", "rv_60",
]

future_cols = ["future_ret_30", "future_ret_60", "future_ret_120"]


def add_future_returns(panel, return_col="ret_1", date_col="date", horizons=(30, 60, 120)):
    panel = panel.copy()

    for h in horizons:
        col = f"future_ret_{h}"

        if col in panel.columns:
            continue

        panel[col] = (
            panel
            .groupby(date_col)[return_col]
            .transform(
                lambda s: s.shift(-1)
                           .rolling(window=h, min_periods=1)
                           .sum()
                           .shift(-(h - 1))
            )
        )

    return panel


def transition_matrix_from_states(states, n_states):
    states = np.asarray(states)
    counts = np.zeros((n_states, n_states), dtype=float)

    prev_states = states[:, :-1].reshape(-1)
    next_states = states[:, 1:].reshape(-1)

    for i, j in zip(prev_states, next_states):
        counts[int(i), int(j)] += 1

    row_sums = counts.sum(axis=1, keepdims=True)

    trans = np.divide(
        counts,
        row_sums,
        out=np.zeros_like(counts),
        where=row_sums != 0,
    )

    return trans


def regime_diagnostics_from_states(states, n_states):
    states = np.asarray(states)
    flat = states.reshape(-1)

    usage = np.array([(flat == k).mean() for k in range(n_states)])

    effective_regimes = np.exp(
        -np.sum(usage * np.log(usage + 1e-12))
    )

    switch_rate = np.mean(states[:, 1:] != states[:, :-1])

    trans = transition_matrix_from_states(states, n_states)

    return {
        "usage": usage,
        "effective_regimes": float(effective_regimes),
        "switch_rate": float(switch_rate),
        "transition_matrix": trans,
        "transition_diag": np.diag(trans),
        "avg_transition_diag": float(np.diag(trans).mean()),
    }


def daily_segment_summary_from_states(states, dates):
    states = np.asarray(states)

    rows = []

    for i in range(states.shape[0]):
        s = states[i]
        switches = int(np.sum(s[1:] != s[:-1]))
        segments = switches + 1

        rows.append({
            "date": pd.to_datetime(dates[i]).date(),
            "switches": switches,
            "segments": segments,
        })

    df = pd.DataFrame(rows)

    summary = {
        "median_segments": float(df["segments"].median()),
        "mean_segments": float(df["segments"].mean()),
        "p75_segments": float(df["segments"].quantile(0.75)),
        "p90_segments": float(df["segments"].quantile(0.90)),
        "max_segments": int(df["segments"].max()),
        "one_state_share": float((df["segments"] == 1).mean()),
        "multi_state_share": float((df["segments"] > 1).mean()),
    }

    return summary, df


def regime_aware_return_diagnostic(
    val_panel,
    test_panel,
    state_col,
    return_col="ret_1",
    date_col="date",
):
    # Choose investable state using validation set only
    val_state_mean_ret = (
        val_panel
        .groupby(state_col)[return_col]
        .mean()
        .sort_index()
    )

    invest_state = int(val_state_mean_ret.idxmax())

    test_tmp = test_panel.copy()
    test_tmp["invest"] = (test_tmp[state_col] == invest_state).astype(float)
    test_tmp["strategy_ret"] = test_tmp["invest"] * test_tmp[return_col]

    daily_strategy_ret = (
        test_tmp
        .groupby(date_col)["strategy_ret"]
        .sum()
    )

    daily_buy_hold_ret = (
        test_tmp
        .groupby(date_col)[return_col]
        .sum()
    )

    regime_std = daily_strategy_ret.std()
    bh_std = daily_buy_hold_ret.std()

    return {
        "invest_state": invest_state,
        "validation_state_mean_returns": val_state_mean_ret,
        "avg_daily_regime_return": float(daily_strategy_ret.mean()),
        "std_daily_regime_return": float(regime_std),
        "avg_daily_buy_hold_return": float(daily_buy_hold_ret.mean()),
        "std_daily_buy_hold_return": float(bh_std),
        "diagnostic_sharpe_regime": float(daily_strategy_ret.mean() / regime_std) if regime_std != 0 else np.nan,
        "diagnostic_sharpe_buy_hold": float(daily_buy_hold_ret.mean() / bh_std) if bh_std != 0 else np.nan,
        "annualized_diagnostic_sharpe_regime": float(np.sqrt(252) * daily_strategy_ret.mean() / regime_std) if regime_std != 0 else np.nan,
        "annualized_diagnostic_sharpe_buy_hold": float(np.sqrt(252) * daily_buy_hold_ret.mean() / bh_std) if bh_std != 0 else np.nan,
        "daily_strategy_returns": daily_strategy_ret,
        "daily_buy_hold_returns": daily_buy_hold_ret,
    }


def compute_model_thesis_outputs(
    states_val,
    states_test,
    val_panel,
    test_panel,
    n_states,
    state_col,
    model_name,
    selected_key=None,
    dates_test=None,
):
    states_val = np.asarray(states_val)
    states_test = np.asarray(states_test)

    val_tmp = val_panel.copy()
    test_tmp = test_panel.copy()

    # Safety checks
    assert len(val_tmp) == states_val.size, (
        f"val_panel rows {len(val_tmp)} do not match states_val size {states_val.size}"
    )
    assert len(test_tmp) == states_test.size, (
        f"test_panel rows {len(test_tmp)} do not match states_test size {states_test.size}"
    )

    val_tmp[state_col] = states_val.reshape(-1)
    test_tmp[state_col] = states_test.reshape(-1)

    # Make sure future-return columns exist
    val_tmp = add_future_returns(val_tmp)
    test_tmp = add_future_returns(test_tmp)

    econ_summary = (
        test_tmp
        .groupby(state_col)[econ_cols]
        .mean()
    )

    future_summary = (
        test_tmp
        .groupby(state_col)[future_cols]
        .agg(["mean", "std", "count"])
    )

    diag = regime_diagnostics_from_states(states_test, n_states=n_states)

    if dates_test is None:
        unique_dates = (
            test_tmp[["date"]]
            .drop_duplicates()
            .sort_values("date")["date"]
            .values
        )
        dates_for_segments = unique_dates
    else:
        dates_for_segments = dates_test

    daily_summary, daily_segments = daily_segment_summary_from_states(
        states_test,
        dates_for_segments
    )

    rar = regime_aware_return_diagnostic(
        val_panel=val_tmp,
        test_panel=test_tmp,
        state_col=state_col,
        return_col="ret_1",
        date_col="date",
    )

    compact = {
        "model": model_name,
        "states": n_states,
        "selected_key": str(selected_key),
        "usage": str(np.round(diag["usage"], 6)),
        "effective_regimes": diag["effective_regimes"],
        "switch_rate": diag["switch_rate"],
        "transition_diag": str(np.round(diag["transition_diag"], 6)),
        "avg_transition_diag": diag["avg_transition_diag"],
        "median_segments": daily_summary["median_segments"],
        "mean_segments": daily_summary["mean_segments"],
        "p75_segments": daily_summary["p75_segments"],
        "p90_segments": daily_summary["p90_segments"],
        "max_segments": daily_summary["max_segments"],
        "one_state_share": daily_summary["one_state_share"],
        "invest_state": rar["invest_state"],
        "avg_daily_regime_return": rar["avg_daily_regime_return"],
        "avg_daily_buy_hold_return": rar["avg_daily_buy_hold_return"],
        "diagnostic_sharpe_regime": rar["diagnostic_sharpe_regime"],
        "diagnostic_sharpe_buy_hold": rar["diagnostic_sharpe_buy_hold"],
        "annualized_diagnostic_sharpe_regime": rar["annualized_diagnostic_sharpe_regime"],
        "annualized_diagnostic_sharpe_buy_hold": rar["annualized_diagnostic_sharpe_buy_hold"],
    }

    compact_df = pd.DataFrame([compact])

    return {
        "val_panel": val_tmp,
        "test_panel": test_tmp,
        "econ_summary": econ_summary,
        "future_summary": future_summary,
        "diagnostics": diag,
        "daily_summary": daily_summary,
        "daily_segments": daily_segments,
        "regime_aware_return": rar,
        "compact_df": compact_df,
    }

print("Helper functions ready.")

Helper functions ready.


In [15]:
import numpy as np
import pandas as pd

# Load panel if needed
if "panel_features" not in globals():
    panel_features = pd.read_parquet(DATA_DIR / "panel_features_4.parquet")

# Load date arrays
dates_train = np.load(DATA_DIR / "dates_train_4.npy", allow_pickle=True)
dates_val   = np.load(DATA_DIR / "dates_val_4.npy", allow_pickle=True)
dates_test  = np.load(DATA_DIR / "dates_test_4.npy", allow_pickle=True)

dates_train = pd.to_datetime(dates_train)
dates_val   = pd.to_datetime(dates_val)
dates_test  = pd.to_datetime(dates_test)

# Prepare panel
panel_features["date"] = pd.to_datetime(panel_features["date"])

panel_features = (
    panel_features
    .sort_values(["date", "m_index"])
    .reset_index(drop=True)
)

# Build validation/test panels
val_panel = (
    panel_features[panel_features["date"].isin(dates_val)]
    .sort_values(["date", "m_index"])
    .reset_index(drop=True)
)

test_panel = (
    panel_features[panel_features["date"].isin(dates_test)]
    .sort_values(["date", "m_index"])
    .reset_index(drop=True)
)

print("val_panel:", val_panel.shape)
print("test_panel:", test_panel.shape)

print("Expected val rows:", states_val_mvae_2.size)
print("Expected test rows:", states_test_mvae_2.size)

assert len(val_panel) == states_val_mvae_2.size
assert len(test_panel) == states_test_mvae_2.size

print("val_panel and test_panel are ready.")

val_panel: (491878, 34)
test_panel: (568123, 34)
Expected val rows: 491878
Expected test rows: 568123
val_panel and test_panel are ready.


In [16]:
selected_key_mvae_2 = (10.0, 10.0, 20.0, 0.7, 5.0)

out_mvae_2 = compute_model_thesis_outputs(
    states_val=states_val_mvae_2,
    states_test=states_test_mvae_2,
    val_panel=val_panel,
    test_panel=test_panel,
    n_states=2,
    state_col="state_mvae_2",
    model_name="Mixture-VAE 2-state",
    selected_key=selected_key_mvae_2,
    dates_test=dates_test,
)

display(out_mvae_2["compact_df"])
display(out_mvae_2["econ_summary"])
display(out_mvae_2["future_summary"])

print("2-state transition matrix:")
print(np.round(out_mvae_2["diagnostics"]["transition_matrix"], 6))

print("\n2-state validation mean returns:")
print(out_mvae_2["regime_aware_return"]["validation_state_mean_returns"])

,model,states,selected_key,usage,effective_regimes,switch_rate,transition_diag,avg_transition_diag,median_segments,mean_segments,...,p90_segments,max_segments,one_state_share,invest_state,avg_daily_regime_return,avg_daily_buy_hold_return,diagnostic_sharpe_regime,diagnostic_sharpe_buy_hold,annualized_diagnostic_sharpe_regime,annualized_diagnostic_sharpe_buy_hold
0,Mixture-VAE 2-state,2,"(10.0, 10.0, 20.0, 0.7, 5.0)",[0.581216 0.418784],1.973674,0.025103,[0.977333 0.971515],0.974424,11.0,10.790089,...,16.0,26,0.002065,0,0.004066,0.000497,0.384524,0.0373,6.104126,0.592127


,ret_mean_30,ret_mean_60,ret_mean_120,momentum_30,momentum_60,momentum_120,trend_strength_30,trend_strength_60,trend_strength_120,ret_std_60,rv_60
state_mvae_2,,,,,,,,,,,
0,0.000037,0.000025,0.000013,0.001111,0.001484,0.001538,0.60837,0.584057,0.446895,0.000269,0.000009
1,-0.000050,-0.000033,-0.000017,-0.001501,-0.001988,-0.002025,-0.68336,-0.633059,-0.441414,0.000360,0.000014


future_ret_30                   future_ret_60                    \
                      mean       std   count          mean       std   count   
state_mvae_2                                                                   
0                 0.000046  0.002451  308043      0.000083  0.003360  283504   
1                -0.000021  0.002704  217943     -0.000032  0.003744  198892   

             future_ret_120                    
                       mean       std   count  
state_mvae_2                                   
0                  0.000102  0.004669  234607  
1                  0.000014  0.005080  160609

2-state transition matrix:
[[0.977333 0.022667]
 [0.028485 0.971515]]

2-state validation mean returns:
state_mvae_2
0    0.000010
1   -0.000013
Name: ret_1, dtype: float64


In [59]:
# SAVE MIXTURE-VAE RESULTS

out_mvae_2["compact_df"].to_csv(
    OUT_DIR / "summary_mvae_2state_metrics.csv",
    index=False
)

out_mvae_2["econ_summary"].to_csv(
    OUT_DIR / "econ_summary_mvae_2.csv"
)

out_mvae_2["future_summary"].to_csv(
    OUT_DIR / "future_summary_mvae_2.csv"
)

out_mvae_2["daily_segments"].to_csv(
    OUT_DIR / "mvae_2state_test_daily_segments.csv",
    index=False
)

out_mvae_2["regime_aware_return"]["daily_strategy_returns"].to_csv(
    OUT_DIR / "mvae_2state_daily_regime_aware_returns.csv"
)

out_mvae_2["regime_aware_return"]["daily_buy_hold_returns"].to_csv(
    OUT_DIR / "mvae_2state_daily_buy_hold_returns.csv"
)

out_mvae_2["test_panel"].to_parquet(
    OUT_DIR / "test_panel_mvae_2state.parquet",
    index=False
)

print("Saved Mixture-VAE individual outputs.")

Saved Mixture-VAE individual outputs.
